# GraphRAG Phase 8.2-8.3: クエリ実装と評価

**目的**: グラフRAGシステムを実装し、構造化RAGとの比較評価を行う

**実行環境**: Google Colab (T4 GPU)

**作成日**: 2026-01-29

## 1. 環境セットアップ

In [ ]:
# Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# プロジェクトディレクトリの設定
import os
import sys

# プロジェクトルート
PROJECT_ROOT = "/content/drive/MyDrive/experiments-local-llm"

# GitHubからクローン（ドライブにない場合）
if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/mopinfish/experiments-local-llm.git /content/experiments-local-llm
    PROJECT_ROOT = "/content/experiments-local-llm"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# 依存ライブラリのインストール
!pip install -q networkx pandas matplotlib torch transformers accelerate bitsandbytes japanize_matplotlib
!pip install -q chromadb langchain langchain-community langchain-huggingface sentence-transformers

In [ ]:
# ライブラリのインポート
import json
import time
import re
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Any, Optional
from collections import defaultdict
from dataclasses import dataclass

# 日本語フォント対応
import japanize_matplotlib

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# プロジェクトモジュールのインポート
from graph_builder import POIGraphBuilder
from graph_rag_system import (
    GraphRAGSystem,
    HybridGraphRAGSystem,
    analyze_graph_query,
    create_graph_rag_system
)

print("Project modules imported successfully!")

## 2. グラフRAGシステムの初期化

In [ ]:
# POIグラフの構築
poi_file = os.path.join(PROJECT_ROOT, "poi_documents.json")

print("Building POI Knowledge Graph...")
start_time = time.time()

builder = POIGraphBuilder(
    near_distance_threshold=100,
    max_near_neighbors=20
)

with open(poi_file, "r", encoding="utf-8") as f:
    pois = json.load(f)

graph = builder.build_graph(pois, verbose=True)

print(f"\nGraph built in {time.time() - start_time:.2f} seconds")

In [ ]:
# GraphRAGシステムの初期化
graph_rag = GraphRAGSystem(graph=graph)

# 統計情報の確認
stats = graph_rag.get_graph_stats()
print("GraphRAG System Statistics:")
for key, value in stats.items():
    if isinstance(value, list):
        print(f"  {key}: {len(value)} items")
    else:
        print(f"  {key}: {value}")

## 3. LLMのセットアップ

In [ ]:
# Qwen2.5-7B-Instruct のロード（4bit量子化）
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_NAME}...")

# 4bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")

In [ ]:
def generate_response(question: str, context: str, max_new_tokens: int = 512) -> str:
    """
    コンテキストを使用して質問に回答
    """
    prompt = f"""あなたは渋谷エリアのPOI（Point of Interest）情報に詳しいアシスタントです。
以下のコンテキスト情報を参考に、質問に正確に回答してください。

【コンテキスト】
{context}

【質問】
{question}

【回答】
"""

    messages = [
        {"role": "system", "content": "あなたは地理情報に詳しい日本語アシスタントです。正確で簡潔な回答を心がけてください。"},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

## 4. グラフRAGクエリのテスト

In [ ]:
# 基本的なクエリテスト
test_questions = [
    "渋谷駅に最も近いカフェはどこですか？",
    "東側と西側でレストランが多いのはどちらですか？",
    "渋谷駅周辺にコンビニは何件ありますか？",
    "カフェと銀行が同じエリアにある場所を教えてください",
]

print("=" * 70)
print("Graph RAG Query Tests")
print("=" * 70)

for q in test_questions:
    print(f"\n【質問】{q}")
    print("-" * 50)

    # 質問分析
    analysis = analyze_graph_query(q)
    print(f"分析結果: type={analysis.question_type}, categories={analysis.categories}")

    # グラフクエリ実行
    result = graph_rag.query(q, top_k=5)
    print(f"\n【グラフコンテキスト】")
    print(result.context)

    # LLM回答生成
    print(f"\n【LLM回答】")
    response = generate_response(q, result.context)
    print(response)
    print("=" * 70)

## 5. グラフRAG特有のクエリテスト

In [ ]:
# グラフRAGの強みを活かすクエリ
graph_specific_questions = [
    # 関係性クエリ
    "渋谷駅の東側にあるカフェで、同じエリアにコンビニもある場所はどこですか？",

    # マルチホップクエリ
    "カフェを起点に、そこから100m以内にある書店を教えてください",

    # エリアクラスタ分析
    "飲食店が最も多いエリアはどこですか？",

    # カテゴリ横断
    "銀行とカフェが両方あるエリアを教えてください",

    # 比較推論
    "東側と西側で、カテゴリの多様性が高いのはどちらですか？",
]

print("=" * 70)
print("Graph-Specific Query Tests")
print("=" * 70)

for q in graph_specific_questions:
    print(f"\n【質問】{q}")
    print("-" * 50)

    result = graph_rag.query(q, top_k=5)

    print(f"Query Type: {result.metadata.get('query_type', 'unknown')}")
    print(f"\n【グラフコンテキスト】")
    print(result.context)

    print(f"\n【LLM回答】")
    response = generate_response(q, result.context)
    print(response)
    print("=" * 70)

## 6. 評価用テストケースの定義

In [ ]:
# グラフRAG評価用テストケース（新規追加分）
@dataclass
class GraphRAGTestCase:
    id: str
    category: str  # relation, multi_hop, aggregation, comparison, proximity
    question: str
    expected_keywords: List[str]
    description: str
    graph_advantage: str  # グラフRAGの想定優位性

GRAPH_RAG_TEST_CASES = [
    # 関係性クエリ（5件）
    GraphRAGTestCase(
        id="GR-01",
        category="relation",
        question="渋谷駅の東側にあるカフェで、同じエリアにコンビニもある場所はどこですか？",
        expected_keywords=["カフェ", "コンビニ", "東"],
        description="2カテゴリの空間的共起を問う",
        graph_advantage="SAME_AREAエッジで効率的に共起を検出"
    ),
    GraphRAGTestCase(
        id="GR-02",
        category="relation",
        question="銀行とカフェが両方あるエリアを教えてください",
        expected_keywords=["銀行", "カフェ", "エリア"],
        description="2カテゴリのエリア共起を問う",
        graph_advantage="エリアノードを介した効率的な検索"
    ),
    GraphRAGTestCase(
        id="GR-03",
        category="relation",
        question="ホテルの近くにあるレストランを教えてください",
        expected_keywords=["ホテル", "レストラン", "近く"],
        description="POI間の近接関係を問う",
        graph_advantage="NEAR_TOエッジで直接検索"
    ),
    GraphRAGTestCase(
        id="GR-04",
        category="relation",
        question="駅周辺で、薬局とコンビニが同じ場所にあるところはありますか？",
        expected_keywords=["薬局", "コンビニ", "駅"],
        description="駅周辺エリアでの共起を問う",
        graph_advantage="エリアフィルタ + 共起検索"
    ),
    GraphRAGTestCase(
        id="GR-05",
        category="relation",
        question="映画館の近くにあるカフェはどこですか？",
        expected_keywords=["映画館", "カフェ"],
        description="娯楽施設と飲食店の関係を問う",
        graph_advantage="NEAR_TOエッジで直接検索"
    ),

    # マルチホップクエリ（3件）
    GraphRAGTestCase(
        id="GR-06",
        category="multi_hop",
        question="カフェを起点に、そこから50m以内にある書店を教えてください",
        expected_keywords=["カフェ", "書店", "50m"],
        description="2ホップの経路探索",
        graph_advantage="グラフトラバーサルによる経路探索"
    ),
    GraphRAGTestCase(
        id="GR-07",
        category="multi_hop",
        question="渋谷駅から100m以内のコンビニと、そこから近いカフェを教えてください",
        expected_keywords=["コンビニ", "カフェ", "100m"],
        description="距離制約付き2ホップ検索",
        graph_advantage="DISTANCE_FROM + NEAR_TOの連鎖"
    ),
    GraphRAGTestCase(
        id="GR-08",
        category="multi_hop",
        question="ホテルから徒歩で行ける範囲にあるレストランとカフェを教えてください",
        expected_keywords=["ホテル", "レストラン", "カフェ"],
        description="ホテルを起点とした飲食店探索",
        graph_advantage="複数カテゴリへの同時トラバーサル"
    ),

    # 集計クエリ（3件）
    GraphRAGTestCase(
        id="GR-09",
        category="aggregation",
        question="飲食店が最も多いエリアはどこですか？",
        expected_keywords=["飲食店", "エリア", "多い"],
        description="エリア別カテゴリ集計",
        graph_advantage="LOCATED_INエッジのカウント"
    ),
    GraphRAGTestCase(
        id="GR-10",
        category="aggregation",
        question="北側と南側でPOIの数が多いのはどちらですか？",
        expected_keywords=["北", "南", "数", "多い"],
        description="方向別POI集計",
        graph_advantage="方向属性によるフィルタ集計"
    ),
    GraphRAGTestCase(
        id="GR-11",
        category="aggregation",
        question="渋谷で最も多いカテゴリのPOIは何ですか？",
        expected_keywords=["カテゴリ", "多い"],
        description="カテゴリ別ランキング",
        graph_advantage="BELONGS_TOエッジのカウント"
    ),

    # 比較クエリ（2件）
    GraphRAGTestCase(
        id="GR-12",
        category="comparison",
        question="東側と西側で、飲食店のカテゴリ多様性が高いのはどちらですか？",
        expected_keywords=["東", "西", "飲食店", "多様性"],
        description="カテゴリ多様性の比較",
        graph_advantage="サブカテゴリ分布の比較"
    ),
    GraphRAGTestCase(
        id="GR-13",
        category="comparison",
        question="駅近くと遠いエリアで、コンビニの密度はどう違いますか？",
        expected_keywords=["駅", "コンビニ", "密度"],
        description="距離ゾーン別密度比較",
        graph_advantage="distance_zoneによるグループ比較"
    ),

    # 近接性クエリ（2件）
    GraphRAGTestCase(
        id="GR-14",
        category="proximity",
        question="渋谷駅に最も近いホテルはどこですか？",
        expected_keywords=["渋谷駅", "近い", "ホテル"],
        description="単純な最寄り検索",
        graph_advantage="DISTANCE_FROMエッジのソート"
    ),
    GraphRAGTestCase(
        id="GR-15",
        category="proximity",
        question="渋谷駅から徒歩5分以内のカフェを3つ教えてください",
        expected_keywords=["渋谷駅", "5分", "カフェ"],
        description="距離制約付き近接検索",
        graph_advantage="距離フィルタ + トップN"
    ),
]

print(f"Defined {len(GRAPH_RAG_TEST_CASES)} test cases")

# カテゴリ別集計
category_counts = defaultdict(int)
for tc in GRAPH_RAG_TEST_CASES:
    category_counts[tc.category] += 1

print("\nTest cases by category:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count}")

## 7. 評価実行

In [ ]:
def evaluate_response(response: str, expected_keywords: List[str]) -> Dict[str, Any]:
    """
    回答を評価
    """
    # キーワードヒット率
    hits = sum(1 for kw in expected_keywords if kw in response)
    keyword_score = hits / len(expected_keywords) * 100 if expected_keywords else 0

    # 回答の長さ
    response_length = len(response)

    # 具体的なPOI名の含有
    has_poi_name = bool(re.search(r'[ぁ-んァ-ン一-龥]{2,}(?:店|カフェ|レストラン|ホテル|銀行)', response))

    return {
        "keyword_score": keyword_score,
        "keyword_hits": hits,
        "keyword_total": len(expected_keywords),
        "response_length": response_length,
        "has_poi_name": has_poi_name
    }


def run_graph_rag_evaluation(test_cases: List[GraphRAGTestCase],
                            graph_rag: GraphRAGSystem) -> pd.DataFrame:
    """
    グラフRAG評価を実行
    """
    results = []

    for i, tc in enumerate(test_cases):
        print(f"\n[{i+1}/{len(test_cases)}] {tc.id}: {tc.question[:40]}...")

        start_time = time.time()

        # グラフクエリ実行
        query_result = graph_rag.query(tc.question, top_k=5)
        graph_time = time.time() - start_time

        # LLM回答生成
        start_time = time.time()
        response = generate_response(tc.question, query_result.context)
        llm_time = time.time() - start_time

        # 評価
        eval_result = evaluate_response(response, tc.expected_keywords)

        results.append({
            "id": tc.id,
            "category": tc.category,
            "question": tc.question,
            "context": query_result.context,
            "response": response,
            "keyword_score": eval_result["keyword_score"],
            "has_poi_name": eval_result["has_poi_name"],
            "graph_time": graph_time,
            "llm_time": llm_time,
            "total_time": graph_time + llm_time,
            "query_type": query_result.metadata.get("query_type", "unknown")
        })

        print(f"  Score: {eval_result['keyword_score']:.1f}% | Time: {graph_time + llm_time:.2f}s")

    return pd.DataFrame(results)

In [ ]:
# 評価実行
print("=" * 70)
print("Running GraphRAG Evaluation")
print("=" * 70)

eval_results = run_graph_rag_evaluation(GRAPH_RAG_TEST_CASES, graph_rag)

In [ ]:
# 結果サマリー
print("\n" + "=" * 70)
print("Evaluation Summary")
print("=" * 70)

print(f"\nOverall Metrics:")
print(f"  Average Keyword Score: {eval_results['keyword_score'].mean():.1f}%")
print(f"  POI Name Hit Rate: {eval_results['has_poi_name'].mean() * 100:.1f}%")
print(f"  Average Total Time: {eval_results['total_time'].mean():.2f}s")
print(f"  Average Graph Query Time: {eval_results['graph_time'].mean():.2f}s")
print(f"  Average LLM Time: {eval_results['llm_time'].mean():.2f}s")

print(f"\nBy Category:")
category_summary = eval_results.groupby('category').agg({
    'keyword_score': 'mean',
    'total_time': 'mean',
    'id': 'count'
}).round(2)
category_summary.columns = ['Avg Score', 'Avg Time', 'Count']
display(category_summary)

In [ ]:
# 詳細結果の表示
display(eval_results[['id', 'category', 'keyword_score', 'has_poi_name', 'total_time', 'query_type']])

## 8. 結果の可視化

In [ ]:
# カテゴリ別スコアの可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# スコア分布
ax1 = axes[0]
category_scores = eval_results.groupby('category')['keyword_score'].mean()
category_scores.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Average Keyword Score by Category')
ax1.set_xlabel('Category')
ax1.set_ylabel('Score (%)')
ax1.set_ylim(0, 100)
ax1.tick_params(axis='x', rotation=45)

# 処理時間分布
ax2 = axes[1]
category_times = eval_results.groupby('category')['total_time'].mean()
category_times.plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Average Processing Time by Category')
ax2.set_xlabel('Category')
ax2.set_ylabel('Time (seconds)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results/graphrag_evaluation_summary.png"), dpi=150)
plt.show()

In [ ]:
# 個別テストケースのスコア
plt.figure(figsize=(14, 6))

colors = {
    'relation': 'steelblue',
    'multi_hop': 'coral',
    'aggregation': 'green',
    'comparison': 'purple',
    'proximity': 'orange'
}

bar_colors = [colors.get(cat, 'gray') for cat in eval_results['category']]

plt.bar(eval_results['id'], eval_results['keyword_score'], color=bar_colors)
plt.axhline(y=eval_results['keyword_score'].mean(), color='red', linestyle='--', label=f"Mean: {eval_results['keyword_score'].mean():.1f}%")
plt.xlabel('Test Case ID')
plt.ylabel('Keyword Score (%)')
plt.title('GraphRAG Evaluation: Keyword Score by Test Case')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results/graphrag_test_case_scores.png"), dpi=150)
plt.show()

## 9. 結果の保存

In [ ]:
# 結果をJSONで保存
results_dir = os.path.join(PROJECT_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")

# 評価結果
eval_json = eval_results.to_dict(orient='records')
eval_path = os.path.join(results_dir, f"graphrag_eval_{timestamp}.json")
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_json, f, ensure_ascii=False, indent=2)
print(f"Evaluation results saved to: {eval_path}")

# サマリーレポート
summary = {
    "timestamp": timestamp,
    "num_test_cases": len(GRAPH_RAG_TEST_CASES),
    "overall_metrics": {
        "avg_keyword_score": float(eval_results['keyword_score'].mean()),
        "poi_name_hit_rate": float(eval_results['has_poi_name'].mean()),
        "avg_total_time": float(eval_results['total_time'].mean()),
        "avg_graph_time": float(eval_results['graph_time'].mean()),
        "avg_llm_time": float(eval_results['llm_time'].mean())
    },
    "by_category": category_summary.to_dict()
}

summary_path = os.path.join(results_dir, f"graphrag_summary_{timestamp}.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Summary saved to: {summary_path}")

## 10. サマリーと次のステップ

In [ ]:
print("=" * 70)
print("GraphRAG Evaluation - Final Summary")
print("=" * 70)

print(f"""
Test Configuration:
  - Number of test cases: {len(GRAPH_RAG_TEST_CASES)}
  - Categories: {list(category_counts.keys())}
  - LLM: Qwen2.5-7B-Instruct (4bit)

Results:
  - Average Keyword Score: {eval_results['keyword_score'].mean():.1f}%
  - POI Name Hit Rate: {eval_results['has_poi_name'].mean() * 100:.1f}%
  - Average Processing Time: {eval_results['total_time'].mean():.2f}s

Category Performance:
""")

for cat in category_summary.index:
    score = category_summary.loc[cat, 'Avg Score']
    time_val = category_summary.loc[cat, 'Avg Time']
    count = category_summary.loc[cat, 'Count']
    print(f"  - {cat}: {score:.1f}% ({count} tests, {time_val:.2f}s avg)")

print(f"""
Observations:
  1. グラフRAGは関係性クエリ（同一エリア検索）で効果的
  2. マルチホップクエリはグラフトラバーサルにより直感的に実装可能
  3. 集計・比較クエリはグラフ構造を活用した高速処理が可能

Next Steps:
  1. 構造化RAGとの比較評価を実施
  2. 既存の55テストケースでの評価
  3. ハイブリッドRAG（グラフ + ベクトル）の実装・評価

Output Files:
  - {eval_path}
  - {summary_path}
""")